Train set (85%); Validation set (15%)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
"""
d2c_v2 dataloader: extract data, grouped 85/15 train/val split, paired-image Dataset.

Set INCLUDE_BENIGN:
    False -> experiment A: negatives are only "no_change" (same-engine re-render)
    True  -> experiment B: negatives are "no_change" + "benign" (Firefox / WebKit renders)
"""
import os
import json
import random
import shutil
import zipfile
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# =====================================================================
# Config
# =====================================================================
ZIP_ON_DRIVE = "/content/drive/MyDrive/CSC490_data/d2c_v3.zip"
LOCAL_ZIP = "/content/d2c_v3.zip"
EXTRACT_TO = "/content"
DATA_ROOT = "/content/d2c_v2"          # top-level folder inside the zip

INCLUDE_BENIGN = True                  # False = experiment A, True = experiment B
DATASET_JSON = "master_dataset_with_benign.json" if INCLUDE_BENIGN else "master_dataset.json"

TRAIN_RATIO = 0.85
SPLIT_SEED = 490

# =====================================================================
# Copy zip to local disk and extract (much faster than reading Drive)
# =====================================================================
if not os.path.exists(os.path.join(DATA_ROOT, "master_dataset_with_benign.json")):
    print("Copying zip to local disk...")
    shutil.copy(ZIP_ON_DRIVE, LOCAL_ZIP)
    print("Extracting...")
    with zipfile.ZipFile(LOCAL_ZIP, "r") as z:
        z.extractall(EXTRACT_TO)
print("Data ready at", DATA_ROOT)

# =====================================================================
# Label maps (match generate_full_dataset.py)
# =====================================================================
FAMILY_MAP = {"typography": 0, "colour": 1, "appearance": 2, "shape": 3, "layout": 4}
OPERATOR_MAP = {
    "font_size": 0, "font_weight": 1, "line_height": 2, "letter_spacing": 3,
    "text_color": 4, "background_color": 5, "opacity": 6,
    "border_radius": 7, "border_width": 8,
    "padding": 9, "margin": 10, "position": 11, "width": 12,
}
IGNORE = -100  # class targets for negatives; use CrossEntropyLoss(ignore_index=-100)


# =====================================================================
# Grouped split: a page and ALL of its variants go to the same set
# =====================================================================
def make_grouped_split(root, train_ratio=TRAIN_RATIO, seed=SPLIT_SEED):
    with open(os.path.join(root, DATASET_JSON), encoding="utf-8") as f:
        recs = json.load(f)
    print(f"loaded {DATASET_JSON}: {len(recs)} pairs")

    pages = sorted({r["parent_id"] for r in recs})
    random.Random(seed).shuffle(pages)
    n_train = int(round(train_ratio * len(pages)))
    train_pages = set(pages[:n_train])

    train_recs = [r for r in recs if r["parent_id"] in train_pages]
    val_recs = [r for r in recs if r["parent_id"] not in train_pages]

    overlap = {r["parent_id"] for r in train_recs} & {r["parent_id"] for r in val_recs}
    assert not overlap, f"leakage: {len(overlap)} pages in both splits"

    with open(os.path.join(root, "splits_85_15.json"), "w") as f:
        json.dump({"train": sorted(train_pages), "val": sorted(set(pages) - train_pages)}, f, indent=2)

    for name, rs in [("train", train_recs), ("val", val_recs)]:
        n_pages = len({r["parent_id"] for r in rs})
        print(f"[{name}] {n_pages} pages | {len(rs)} pairs | labels: {dict(Counter(r['label'] for r in rs))}")
    return train_recs, val_recs


# =====================================================================
# Dataset
# =====================================================================
class PairedUIRegressionDataset(Dataset):
    """
    Returns (reference, candidate) image pairs.

    mode="tile": crop the SAME window from both images at native resolution.
        - regression: window placed around the mutated element (bbox), with random jitter
        - negatives : random window inside the height both images share
    mode="full": resize the whole page to full_size (small changes may disappear).
    """

    def __init__(self, records, root, mode="tile", tile=512, full_size=(448, 448),
                 jitter=True, transform=None):
        self.records = records
        self.root = root
        self.mode = mode
        self.tile = tile
        self.full_size = full_size
        self.jitter = jitter
        self.transform = transform or transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.records)

    @staticmethod
    def _pad_to(img, W, H):
        """Pad with white so both images share the same canvas."""
        if img.size == (W, H):
            return img
        canvas = Image.new("RGB", (W, H), "white")
        canvas.paste(img, (0, 0))
        return canvas

    def _target_box(self, r):
        """Union of bbox before/after, i.e. where the change lives."""
        b0, b1 = r.get("bbox_before"), r.get("bbox_after")
        if not b0 or not b1:
            return None
        x0 = min(b0[0], b1[0]); y0 = min(b0[1], b1[1])
        x1 = max(b0[0] + b0[2], b1[0] + b1[2]); y1 = max(b0[1] + b0[3], b1[1] + b1[3])
        return x0, y0, x1, y1

    def _choose_window(self, r, W, H, H_common):
        T = self.tile
        box = self._target_box(r) if r["label"] == "regression" else None
        if box is None:
            # negatives: stay inside the height both images actually have, so a tile never
            # contains the white padding added to the shorter page (a trivial shortcut)
            x = random.randint(0, max(0, W - T))
            y = random.randint(0, max(0, min(H, H_common) - T))
            return x, y
        cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
        if self.jitter:  # so the change is not always dead-centre
            cx += random.uniform(-T / 4, T / 4)
            cy += random.uniform(-T / 4, T / 4)
        x = int(min(max(0, cx - T / 2), max(0, W - T)))
        y = int(min(max(0, cy - T / 2), max(0, H - T)))
        return x, y

    def __getitem__(self, idx):
        r = self.records[idx]
        ref = Image.open(os.path.join(self.root, r["reference_path"])).convert("RGB")
        cand = Image.open(os.path.join(self.root, r["candidate_path"])).convert("RGB")

        H_common = min(ref.height, cand.height)   # measured BEFORE padding
        W, H = max(ref.width, cand.width), max(ref.height, cand.height)
        ref, cand = self._pad_to(ref, W, H), self._pad_to(cand, W, H)

        if self.mode == "tile":
            T = self.tile
            if W < T or H < T:  # tiny page: pad up to one tile
                W2, H2 = max(W, T), max(H, T)
                ref, cand = self._pad_to(ref, W2, H2), self._pad_to(cand, W2, H2)
                W, H = W2, H2
            x, y = self._choose_window(r, W, H, H_common)
            ref = ref.crop((x, y, x + T, y + T))
            cand = cand.crop((x, y, x + T, y + T))
        else:
            ref = ref.resize(self.full_size, Image.BILINEAR)
            cand = cand.resize(self.full_size, Image.BILINEAR)

        is_reg = r["label"] == "regression"
        return {
            "img_ref": self.transform(ref),
            "img_cand": self.transform(cand),
            "is_regression": torch.tensor(1.0 if is_reg else 0.0),
            "family": torch.tensor(FAMILY_MAP[r["family"]] if is_reg else IGNORE),
            "operator": torch.tensor(OPERATOR_MAP[r["operator"]] if is_reg else IGNORE),
            "sample_id": r["sample_id"],
        }


# =====================================================================
# Loaders + class balance
# =====================================================================
train_recs, val_recs = make_grouped_split(DATA_ROOT)

train_ds = PairedUIRegressionDataset(train_recs, DATA_ROOT, mode="tile", jitter=True)
val_ds = PairedUIRegressionDataset(val_recs, DATA_ROOT, mode="tile", jitter=False)

# negatives (no_change + benign): ~8% of pairs without benign, ~20% with benign.
# pass this to BCEWithLogitsLoss(pos_weight=...)
n_pos = sum(r["label"] == "regression" for r in train_recs)
n_neg = len(train_recs) - n_pos
pos_weight = torch.tensor(n_neg / max(1, n_pos))
print(f"train pos={n_pos} neg={n_neg} -> pos_weight={pos_weight.item():.3f}")

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

Copying zip to local disk...
Extracting...
Data ready at /content/d2c_v2
loaded master_dataset_with_benign.json: 7271 pairs
[train] 411 pages | 6188 pairs | labels: {'no_change': 411, 'regression': 4955, 'benign': 822}
[val] 73 pages | 1083 pairs | labels: {'no_change': 73, 'regression': 864, 'benign': 146}
train pos=4955 neg=1233 -> pos_weight=0.249


In [3]:
batch = next(iter(train_loader))
print("ref  batch:", tuple(batch["img_ref"].shape))
print("cand batch:", tuple(batch["img_cand"].shape))
print("is_regression:", batch["is_regression"].tolist())
print("family:", batch["family"].tolist())
print("sample_id:", batch["sample_id"][:6])

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ref  batch: (16, 3, 512, 512)
cand batch: (16, 3, 512, 512)
is_regression: [1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0]
family: [0, -100, 0, -100, -100, 0, 0, 0, 4, 0, 3, 1, 0, -100, 0, -100]
sample_id: ['d2c_227_letter_spacing', 'd2c_380_firefox', 'd2c_385_font_size', 'd2c_251_webkit', 'd2c_480_firefox', 'd2c_416_letter_spacing']


In [4]:
!cp /content/d2c_v2/splits_85_15.json /content/drive/MyDrive/CSC490_data/

In [5]:
train_pages = {r["parent_id"] for r in train_recs}
val_pages = {r["parent_id"] for r in val_recs}
print("pages in both sets:", len(train_pages & val_pages))   # expect 0

# example: every variant of one train page is in train
pid = sorted(train_pages)[0]
print(pid, "in train:", sorted(r["sample_id"] for r in train_recs if r["parent_id"] == pid))
print(pid, "in val:  ", [r["sample_id"] for r in val_recs if r["parent_id"] == pid])   # expect []

pages in both sets: 0
d2c_000 in train: ['d2c_000_background_color', 'd2c_000_border_width', 'd2c_000_firefox', 'd2c_000_font_size', 'd2c_000_font_weight', 'd2c_000_identical', 'd2c_000_letter_spacing', 'd2c_000_line_height', 'd2c_000_margin', 'd2c_000_opacity', 'd2c_000_padding', 'd2c_000_position', 'd2c_000_text_color', 'd2c_000_webkit', 'd2c_000_width']
d2c_000 in val:   []
